In [6]:
import pandas as pd
import re
from collections import Counter


In [2]:

resume_df = pd.read_csv("../data/processed/resumes_clean.csv")
skills_df = pd.read_csv("../data/processed/skills_clean.csv")

print("Resumes:", len(resume_df))
print("Skills:", len(skills_df))

Resumes: 2484
Skills: 36167


In [3]:
resume_text = resume_df["clean_resume"].fillna("")
skill_vocabulary = skills_df["clean_skill"].tolist()

print("Total resumes:", len(resume_text))
print("Total skills:", len(skill_vocabulary))

Total resumes: 2484
Total skills: 36167


In [5]:
# Combine all resumes into one text
all_resumes_text = " ".join(resume_text)

print("Combined resume text created.")
print("Total characters:", len(all_resumes_text))

Combined resume text created.
Total characters: 14712248


In [8]:
test_vocabulary = skill_vocabulary[:500]

print("Testing with:", len(test_vocabulary), "skills")

Testing with: 500 skills


In [9]:
test_counts = {}

for skill in test_vocabulary:
    pattern = r"(?<!\w)" + re.escape(skill) + r"(?!\w)"
    count = len(re.findall(pattern, all_resumes_text, flags=re.IGNORECASE))
    test_counts[skill] = count

print("Test completed!")

Test completed!


In [10]:
test_results = pd.DataFrame(
    test_counts.items(),
    columns=["clean_skill", "resume_count"]
)

test_results = test_results.sort_values(
    "resume_count",
    ascending=False
)

test_results.head(30)

,clean_skill,resume_count
97,maintenance,1824
299,cash,1685
438,press releases,209
263,computer science,183
269,french,127
87,information system,99
41,ethics,81
76,prototype,74
35,startup,68
388,prospect,63


In [11]:
important_skills = [
    "python",
    "java",
    "javascript",
    "c++",
    "c#",
    "sql",
    "html",
    "css",
    "react",
    "react.js",
    "node.js",
    "mongodb",
    "mysql",
    "postgresql",
    "numpy",
    "pandas",
    "scikit-learn",
    "tensorflow",
    "pytorch",
    "machine learning",
    "deep learning",
    "artificial intelligence",
    "data analysis",
    "data analytics",
    "power bi",
    "tableau",
    "git",
    "github"
]

for skill in important_skills:
    matches = skills_df[
        skills_df["clean_skill"].str.lower() == skill.lower()
    ]

    if len(matches) > 0:
        print(f"✓ {skill}")
    else:
        print(f"✗ {skill}")

✓ python
✓ java
✓ javascript
✓ c++
✓ c#
✓ sql
✓ html
✓ css
✗ react
✓ react.js
✓ node.js
✓ mongodb
✓ mysql
✓ postgresql
✓ numpy
✓ pandas
✓ scikit-learn
✓ tensorflow
✓ pytorch
✓ machine learning
✓ deep learning
✓ artificial intelligence
✓ data analysis
✓ data analytics
✓ power bi
✓ tableau
✓ git
✓ github


In [12]:
test_skills = [
    "python",
    "java",
    "javascript",
    "c++",
    "c#",
    "sql",
    "html",
    "css",
    "react.js",
    "node.js",
    "mongodb",
    "mysql",
    "postgresql",
    "numpy",
    "pandas",
    "scikit-learn",
    "tensorflow",
    "pytorch",
    "machine learning",
    "deep learning",
    "artificial intelligence",
    "data analysis",
    "data analytics",
    "power bi",
    "tableau",
    "git",
    "github"
]

print("Test vocabulary size:", len(test_skills))

Test vocabulary size: 27


In [ ]:
# Testing on sample resume

In [13]:
sample_resume = """
I am a Data Analyst with experience in Python, Pandas, NumPy and SQL.
I have worked on machine learning projects using Scikit-learn and
TensorFlow. I also have experience with Power BI, Tableau, Git and GitHub.
"""

sample_resume = sample_resume.lower()

print(sample_resume)


i am a data analyst with experience in python, pandas, numpy and sql.
i have worked on machine learning projects using scikit-learn and
tensorflow. i also have experience with power bi, tableau, git and github.



In [14]:
def extract_skills(text, skills):
    text = text.lower()
    found_skills = []

    for skill in skills:
        pattern = r"(?<!\w)" + re.escape(skill.lower()) + r"(?!\w)"

        if re.search(pattern, text):
            found_skills.append(skill)

    return found_skills

In [15]:
detected_skills = extract_skills(
    sample_resume,
    test_skills
)

print("Detected skills:")
print(detected_skills)

Detected skills:
['python', 'sql', 'numpy', 'pandas', 'scikit-learn', 'tensorflow', 'machine learning', 'power bi', 'tableau', 'git', 'github']


In [16]:
# Testing on Real Resume

In [17]:
real_resume = resume_df["clean_resume"].iloc[0]

print(real_resume[:2000])

hr administrator/marketing associate hr administrator summary dedicated customer service manager with 15+ years of experience in hospitality and customer service management. respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service. highlights focused on customer satisfaction team management marketing savvy conflict resolution techniques training and development skilled multi-tasker client relations specialist accomplishments missouri dot supervisor training certification certified by ihg in customer loyalty and marketing by segment hilton worldwide general manager training certification accomplished trainer for cross server hospitality systems such as hilton onq , micros opera pms , fidelio opera reservation system (ors) , holidex completed courses and seminars in customer service, sales strategies, inventory control, loss prevention, safety, time management, leadership and performance assessment. experience hr adm

In [18]:
real_detected_skills = extract_skills(
    real_resume,
    test_skills
)

print("Detected skills:")
print(real_detected_skills)

Detected skills:
['data analysis']


In [19]:
# Building extractor for Resumes

In [20]:
# Create a set of all normalized skills
skill_set = set(skill_vocabulary)

print("Total skills in lookup:", len(skill_set))

Total skills in lookup: 36167


In [22]:
# Create a faster extractor

In [23]:
def extract_skills_fast(text, skill_set):
    text = text.lower()
    
    # Keep technical characters such as +, #, ., -
    text = re.sub(r'[^a-z0-9+#.\- ]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    words = text.split()
    
    found_skills = set()
    
    # Check phrases from longest to shortest
    max_words = 5
    
    for n in range(max_words, 0, -1):
        for i in range(len(words) - n + 1):
            phrase = " ".join(words[i:i+n])
            
            if phrase in skill_set:
                found_skills.add(phrase)
    
    return sorted(found_skills)

In [24]:
# Testing on Real resumes

In [26]:
real_detected_skills = extract_skills_fast(
    real_resume,
    skill_set
)

print("Detected skills:")
print(real_detected_skills)

Detected skills:
['2004', '2007', '2010', '2012', '2013', 'accounting', 'administration', 'advanced', 'advertising', 'analysis', 'analytical', 'analytical skills', 'billing', 'budgeting', 'certification', 'codes', 'collateral', 'compensation', 'computer skills', 'conflict', 'conflict resolution', 'customer', 'customer loyalty', 'customer satisfaction', 'customer service', 'daily operations', 'data', 'data analysis', 'dental insurance', 'design', 'documentation', 'employee relations', 'executive support', 'facebook', 'features', 'financial management', 'front office', 'front-end', 'general', 'government', 'government relations', 'health', 'healthcare', 'holidex', 'hospital', 'hospitality', 'hotel', 'human resources', 'icd-9', 'insurance', 'interpret', 'inventory', 'inventory control', 'labor relations', 'layout', 'leadership', 'leave of absence', 'loss prevention', 'management', 'management accounting', 'marketing', 'medical billing', 'medical terminology', 'micros', 'morale', 'office',

In [27]:
skills_extraction = skills_df.copy()

skills_extraction = skills_extraction[
    ~skills_extraction["clean_skill"].str.fullmatch(r"\d+")
].reset_index(drop=True)

print("Original skills:", len(skills_df))
print("After removing numeric entries:", len(skills_extraction))

Original skills: 36167
After removing numeric entries: 36149


In [28]:
protected_skills = {
    "python",
    "java",
    "javascript",
    "c",
    "c++",
    "c#",
    "f#",
    "g++",
    "sql",
    "html",
    "css",
    "php",
    "ruby",
    "swift",
    "kotlin",
    "go",
    "r",
    "matlab",
    "scala",
    "perl",
    "bash",
    "powershell",
    "git",
    "github",
    "gitlab",
    "npm",
    "node.js",
    "react.js",
    "vue",
    "angular",
    "mongodb",
    "mysql",
    "postgresql",
    "oracle",
    "redis",
    "numpy",
    "pandas",
    "tensorflow",
    "pytorch",
    "keras",
    "tableau",
    "power bi",
    "docker",
    "kubernetes",
    "aws",
    "azure",
    "gcp",
    "linux",
    "android",
    "ios",
    "sap",
    "excel",
    "jira"
}

print("Protected skills:", len(protected_skills))

Protected skills: 53


In [29]:
generic_words = {
    "advanced",
    "analysis",
    "analytical",
    "codes",
    "collateral",
    "conflict",
    "customer",
    "data",
    "design",
    "features",
    "general",
    "government",
    "health",
    "hospital",
    "hotel",
    "insurance",
    "interpret",
    "layout",
    "management",
    "office",
    "operations",
    "paperwork",
    "participation",
    "plan",
    "policies",
    "prevention",
    "print",
    "public",
    "ratings",
    "records",
    "reporting",
    "reviews",
    "safety",
    "sales",
    "service",
    "skills",
    "statistics",
    "system",
    "teams",
    "terminology",
    "training",
    "transactions",
    "website",
    "customer",
    "organization",
    "professional",
    "responsibilities",
    "experience",
    "work",
    "working"
}

print("Generic words:", len(generic_words))

Generic words: 49


In [30]:
def is_valid_extraction_skill(skill):
    skill = skill.lower().strip()

    # Keep multi-word skills
    if len(skill.split()) > 1:
        return True

    # Keep protected technical skills
    if skill in protected_skills:
        return True

    # Remove generic standalone words
    if skill in generic_words:
        return False

    return True

In [31]:
skills_extraction = skills_extraction[
    skills_extraction["clean_skill"].apply(is_valid_extraction_skill)
].reset_index(drop=True)

print("Final extraction vocabulary:", len(skills_extraction))

Final extraction vocabulary: 36106
